# Membuat SparkSession & Dataset
Seperti biasa, jalankan cell ini terlebih dahulu di setiap sesi Jupyter baru.

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [13]:
import numpy as np
import pandas as pd

np.random.seed(7)
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/master: target & manager per kategori (data ini relatif statis, jarang berubah)
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel transaksi: data yang terus bertambah setiap hari
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("df_produk:")
df_produk.show()
print("df_transaksi (5 baris pertama dari total", df_transaksi.count(), "baris):")
df_transaksi.show(5)

df_produk:


+--------------------+--------------+-------+
|            kategori|target_bulanan|manager|
+--------------------+--------------+-------+
|          Elektronik|      50000000|   Andi|
|             Fashion|      40000000|   Budi|
|   Makanan & Minuman|      30000000|  Citra|
|Kesehatan & Kecan...|      25000000|   Dewi|
|        Rumah Tangga|      20000000|    Eka|
+--------------------+--------------+-------+

df_transaksi (5 baris pertama dari total 300 baris):
+--------+--------------------+--------+----------+
|order_id|            kategori|    kota|pendapatan|
+--------+--------------------+--------+----------+
|      O0|        Rumah Tangga|Magelang|    488643|
|      O1|             Fashion|Magelang|    401943|
|      O2|Kesehatan & Kecan...|    Solo|    452308|
|      O3|Kesehatan & Kecan...|Semarang|    421741|
|      O4|        Rumah Tangga|Semarang|    185244|
+--------+--------------------+--------+----------+
only showing top 5 rows



# Latihan Mandiri
Jalankan ulang cell pembuatan SparkSession dan kedua DataFrame (df_transaksi, df_produk) dari awal modul sebelum mengerjakan latihan berikut.

In [14]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

np.random.seed(7)
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")
print("Siap untuk latihan.")

Siap untuk latihan.


### Soal 1
Menggunakan DataFrame API (join), tampilkan seluruh transaksi kota "Solo" beserta nama manager dari kategorinya masing-masing.

In [3]:
# Jawaban Soal 1 di sini
df_solo = df_transaksi.filter(col("kota") == "Solo").join(df_produk, on="kategori", how="left").select("order_id", "kategori", "kota", "pendapatan", "manager")

print("Jumlah transaksi Solo:", df_solo.count())
df_solo.show(10)

NameError: name 'df_transaksi' is not defined

### Soal 2
Menggunakan window function, tentukan transaksi dengan pendapatan tertinggi (peringkat 1 saja) di setiap kota (bukan kategori). tidak boleh menggunakan rank().

In [ ]:
# Jawaban Soal 2 di sini
window_kota = Window.partitionBy("kota").orderBy(col("pendapatan").desc())

df_top_kota = df_transaksi.withColumn("urutan", row_number().over(window_kota)).filter(col("urutan") == 1)
df_top_kota.show()

### Soal 3
Menggunakan Spark SQL (bukan DataFrame API), tulis kueri untuk menghitung rata-rata pendapatan per kota, urutkan dari tertinggi.

In [ ]:
# Jawaban Soal 3 di sini
hasil_soal3 = spark.sql('''
    SELECT kota, AVG(pendapatan) AS rata_rata_pendapatan
    FROM transaksi
    GROUP BY kota
    ORDER BY rata_rata_pendapatan DESC
''')
hasil_soal3.show()

### Soal 4 (Refleksi singkat).
##### Dalam 2-3 kalimat: menurut anda, dalam situasi seperti apa anda akan lebih memilih menulis Spark SQL dibanding DataFrame API pada pekerjaan anda nanti? Tulis jawaban pada markdown cell di bawah ini.

*Memilih spark SQL ketika pekerjaannya berupa join dan agregasi, misalnya laporan rutin untuk tim yang terbiasa dengan SQL. Kueri SQL lebih ringkas dan orang lain mudah membacanya, dan performanya sama karena spark memprosesnya dengan mesin optimisasi yang sama. Untuk transformasi bertahap yang memuat percabangan atau fungsi buatan sendiri mending memakai dataframe API.*

# Tugas Mandiri

### Membuat SparkSession

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round, row_number
from pyspark.sql.window import Window
import numpy as np
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession aktif, versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 00:33:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession aktif, versi Spark: 3.5.9


### Menyiapkan Dataset

In [5]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/xiuviu/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/xiuviu/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/xiuviu/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/xiuviu/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


### Membaca dan Eksplorasi Awal

In [ ]:
path_hdfs = "hdfs://localhost:9000/user/xiuviu/tugas5/transaksi_tugas5.csv"

df_transaksi = spark.read.csv(path_hdfs, header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.printSchema()
df_transaksi.show(5)
df_target.show()

### Join

In [ ]:
ringkasan_kota = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))
hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner").withColumn("pencapaian_persen", spark_round(col("total_pendapatan") / col("target_bulanan") * 100, 2))

hasil_a.orderBy(col("pencapaian_persen").desc()).show()

### Window function

In [ ]:
per_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))

window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())
hasil_b = per_kota_kategori.withColumn("urutan", row_number().over(window_kota)).filter(col("urutan") == 1).orderBy("kota")

hasil_b.show()

### Spark sql

In [ ]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

hasil_c = spark.sql('''
    SELECT t.kota, p.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang p ON t.kota = p.kota
    GROUP BY t.kota, p.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_c.show()

### Kesimpulan
Purworejo berkinerja paling baik. Cabang ini mencatat pendapatan Rp45.650.000 terhadap target Rp30.000.000, sehingga pencapaiannya 152,17% dan menjadi satu-satunya cabang yang melampaui target. Purworejo juga memiliki transaksi terbanyak, 116 transaksi, dengan kategori terlaris Kesehatan & Kecantikan sebesar Rp10.075.000. Target Purworejo paling rendah di antara lima cabang, dan itu ikut menaikkan persentasenya.
Semarang paling perlu perhatian manajemen. Pencapaiannya 69,41% (Rp38.175.000 dari target Rp55.000.000), terendah di antara semua cabang, dengan selisih Rp16.825.000 dari target. Magelang menyusul di 70,33% (Rp31.650.000 dari Rp45.000.000). Yogyakarta membukukan pendapatan absolut tertinggi, Rp47.275.000, tetapi targetnya Rp60.000.000 sehingga pencapaiannya 78,79%. Solo mencapai 83,69%. Manajemen dapat mulai dari Semarang dan Magelang dengan memperkuat kategori terlaris masing-masing: Rumah Tangga di Semarang (Rp11.125.000) dan Kesehatan & Kecantikan di Magelang (Rp7.275.000).